# Activity-Level Analysis within Process Paths (PM4Py + ipywidgets)

Path-level analysis (comparing the alternate **variants** a case can traverse) tells you *where* improvement opportunities lie.
Activity-level analysis **within a path** tells you *how the work is actually done* in each activity of that path.

This notebook loads a digital event log, discovers every path (variant) that case IDs have followed, and lets you pick any path
(or the whole log) and interrogate its activities through six questions:

| # | Question | How it is measured |
|---|----------|--------------------|
| 1 | **Which activities occur most often?** | Executions, % of cases containing the activity, executions per case, share of all events |
| 2 | **Which activities take longest?** | Processing time = complete − start timestamp (mean, median, P90, max, total, variability) |
| 3 | **Which activities cause waiting?** | Waiting *before* an activity (queue in front of it) and waiting *after* it (idle time it hands to the next step), plus the costliest hand-overs |
| 4 | **Which activities repeat?** | Rework rate (% of cases executing the activity more than once), extra executions, self-loops |
| 5 | **Which activities are costly?** | Cost column from the log, or an estimate = fixed cost per execution + hourly rate × processing time; includes rework cost |
| 6 | **Which activities are linked to partial cases?** | Lift of partial-case probability given the activity, drop-off points, and cases that abandoned the selected path part-way |

A **Scorecard** tab ranks every activity across all six dimensions, and a **Path comparison** tab shows how the same activity behaves in different paths.

### How to use
1. Run all cells (*Run ▸ Run All*).
2. **Step 1** – point to your log (`.xes`, `.xes.gz`, `.csv`, `.xlsx`, `.parquet`) by path or upload, or tick the demo log. Click **Load log**.
3. **Step 2** – confirm the column mapping (case ID, activity, timestamps, optional lifecycle / resource / cost). Click **Prepare event log**.
4. **Step 3** – define what a *complete* case looks like (expected start and end activities). Click **Apply & build dashboard**.
5. **Step 4** – use the dashboard: choose a path, time unit, statistic and cost assumptions; switch tabs for each question.

> **Timestamps matter.** Processing time and a clean separation of *working* vs *waiting* need a start timestamp per event
> (a start-time column, or `lifecycle:transition` start/complete pairs). With only one timestamp per event, the notebook still works but
> "waiting before" becomes *elapsed time since the previous event* (waiting + processing), and processing time is unavailable.

## 0 · Imports and shared state

In [1]:
# Run this cell first. It may take 1-2 minutes the first time — that's normal.
import sys, subprocess

def _pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for _pkg in ["pm4py", "ipywidgets", "pandas", "graphviz"]:
    _pip_install(_pkg)

# Graphviz's drawing engine (the "dot" program) needs to exist on the system,
# not just the Python wrapper. This installs it on Debian/Ubuntu-based
# notebook environments (e.g. Google Colab); it's a no-op elsewhere and is
# safe to ignore if it fails (e.g. on Windows or if it's already installed).
try:
    subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=False,
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except Exception:
    pass

print("✅ Setup complete — you can now run the cells below.")

✅ Setup complete — you can now run the cells below.


In [2]:
import io, os, re, tempfile, warnings
import numpy as np
import pandas as pd
import pm4py
import ipywidgets as W
from IPython.display import display, HTML, clear_output, Image
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

STATE = {}          # holds raw log, prepared event table, case table, variants, settings
ALL = "__ALL__"     # dropdown value meaning "all paths / whole log"
UNITS = {"minutes": 60.0, "hours": 1.0, "days": 1 / 24}

print(f"pm4py {pm4py.__version__} | pandas {pd.__version__} | ipywidgets {W.__version__}")



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




pm4py 2.7.23.8 | pandas 2.2.3 | ipywidgets 7.7.1


## 0b · Demo log (optional)
A synthetic loan-application log with start/complete timestamps, resources, costs, rework loops, an approval branch, a rejection branch,
abandoned cases and still-open cases. Use it to explore the notebook before loading your own log.

In [3]:
def make_demo_log(n_cases=600, seed=7):
    rng = np.random.default_rng(seed)
    # activity: (mean work minutes, mean wait hours before it, role, hourly rate)
    acts = {
        "Receive Application":  (10,  0.0, "Clerk",   30),
        "Check Documents":      (25,  4.0, "Clerk",   30),
        "Request Missing Info": (15,  2.0, "Clerk",   30),
        "Credit Assessment":    (60, 12.0, "Analyst", 60),
        "Manager Approval":     (20, 30.0, "Manager", 120),
        "Send Offer":           (15,  3.0, "Clerk",   30),
        "Reject Application":   (10,  2.0, "Analyst", 60),
        "Close Case":           (5,   6.0, "Clerk",   30),
    }
    R, C, M, CA, MA, SO, RJ, CL = ("Receive Application", "Check Documents", "Request Missing Info",
                                   "Credit Assessment", "Manager Approval", "Send Offer",
                                   "Reject Application", "Close Case")
    paths = [([R, C, CA, SO, CL], .45),
             ([R, C, M, C, CA, SO, CL], .20),
             ([R, C, CA, MA, SO, CL], .18),
             ([R, C, CA, RJ, CL], .12),
             ([R, C, M, C, M, C, CA, MA, SO, CL], .05)]
    p = np.array([w for _, w in paths]); p = p / p.sum()
    t0 = pd.Timestamp("2026-01-05 08:00")
    horizon = t0 + pd.Timedelta(days=112)          # log extraction moment -> creates open cases
    rows = []
    for i in range(n_cases):
        seq = list(paths[rng.choice(len(paths), p=p)][0])
        if M in seq and rng.random() < 0.25:        # customers often drop out after info requests
            seq = seq[: seq.index(M) + 1]
        elif rng.random() < 0.05:
            seq = seq[: int(rng.integers(2, len(seq)))]
        t = t0 + pd.Timedelta(days=float(rng.uniform(0, 110)))
        seen = {}
        for k, a in enumerate(seq):
            work, wait, role, rate = acts[a]
            seen[a] = seen.get(a, 0) + 1
            if k > 0:
                t += pd.Timedelta(hours=float(rng.exponential(wait)))
            if a == C and seen[a] > 1:
                work *= 0.6
            d = float(rng.lognormal(np.log(work), 0.4))
            start, end = t, t + pd.Timedelta(minutes=d)
            if end > horizon:
                break
            rows.append({"case_id": f"L{i+1:04d}", "activity": a, "start_time": start, "end_time": end,
                         "resource": f"{role}_{int(rng.integers(1, 4))}",
                         "cost": round(rate * d / 60 + rng.normal(2, 0.5), 2)})
            t = end
    return pd.DataFrame(rows)

## 0c · Loading and preparation functions

In [4]:
def _to_datetime(s):
    """Robust conversion to naive datetime64."""
    if pd.api.types.is_datetime64_any_dtype(s):
        out = s
    else:
        try:
            out = pd.to_datetime(s, errors="coerce", utc=True, format="mixed")
        except (TypeError, ValueError):
            out = pd.to_datetime(s, errors="coerce", utc=True)
    if getattr(out.dt, "tz", None) is not None:
        out = out.dt.tz_convert(None)
    return out


def _uploaded_file(upload_widget):
    """Return (name, bytes) from a FileUpload widget (ipywidgets 7 and 8)."""
    v = upload_widget.value
    if isinstance(v, dict):                      # ipywidgets 7
        name, meta = next(iter(v.items()))
        return name, bytes(meta["content"])
    f = v[0]                                     # ipywidgets 8
    return f["name"], bytes(f["content"])


def load_log(name, content=None, sep=","):
    """Read an event log into a pandas DataFrame. XES is read with pm4py."""
    low = name.lower()
    src = io.BytesIO(content) if content is not None else name
    if low.endswith(".xes") or low.endswith(".xes.gz"):
        path = name
        if content is not None:
            suffix = ".xes.gz" if low.endswith(".gz") else ".xes"
            with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
                f.write(content)
                path = f.name
        log = pm4py.read_xes(path)
        return log if isinstance(log, pd.DataFrame) else pm4py.convert_to_dataframe(log)
    if low.endswith((".csv", ".txt", ".tsv")):
        return pd.read_csv(src, sep=sep)
    if low.endswith((".xlsx", ".xls")):
        return pd.read_excel(src)
    if low.endswith(".parquet"):
        return pd.read_parquet(src)
    raise ValueError("Unsupported file type. Use .xes, .xes.gz, .csv, .xlsx or .parquet")


def guess_mapping(cols):
    low = {c.lower(): c for c in cols}
    def find(exact, contains=(), exclude=()):
        for c in exact:
            if c.lower() in low:
                return low[c.lower()]
        for c in cols:
            cl = c.lower()
            if any(k in cl for k in contains) and not any(x in cl for x in exclude):
                return c
        return None
    return {
        "case": find(["case:concept:name", "case_id", "caseid", "case id", "case"], ("case",)) or cols[0],
        "activity": find(["concept:name", "activity", "activity_name", "event", "task"], ("activ",)) or cols[min(1, len(cols) - 1)],
        "timestamp": find(["time:timestamp", "end_time", "complete_timestamp", "end_timestamp", "timestamp",
                           "end", "completed"], ("time", "date"), ("start",)) or cols[-1],
        "start": find(["start_timestamp", "start_time", "time:start", "start"], ("start",)),
        "lifecycle": find(["lifecycle:transition", "lifecycle", "transition", "event_type"]),
        "resource": find(["org:resource", "resource", "user", "performer"], ("resource",)),
        "cost": find(["cost:total", "cost", "amount", "cost_amount"], ("cost",)),
    }


def prepare_events(raw, m):
    """Build one row per activity instance with start/end, waiting, repetition and hand-over features."""
    raw = raw.loc[:, ~raw.columns.duplicated()].reset_index(drop=True)
    df = pd.DataFrame({
        "case": raw[m["case"]].astype(str),
        "activity": raw[m["activity"]].astype(str),
        "end": _to_datetime(raw[m["timestamp"]]),
    })
    df["start"] = _to_datetime(raw[m["start"]]) if m.get("start") else pd.NaT
    df["resource"] = raw[m["resource"]].astype(str) if m.get("resource") else None
    df["cost_log"] = pd.to_numeric(raw[m["cost"]], errors="coerce") if m.get("cost") else np.nan

    # lifecycle start/complete pairing (only when no explicit start column is mapped)
    if m.get("lifecycle") and not m.get("start"):
        lc = raw[m["lifecycle"]].astype(str).str.lower().str.strip()
        starts = df[lc.eq("start").values].sort_values("end").copy()
        comps = df[lc.isin(["complete", "completed", "end"]).values].sort_values("end").copy()
        if len(comps):
            if len(starts):
                starts["occ"] = starts.groupby(["case", "activity"]).cumcount()
                comps["occ"] = comps.groupby(["case", "activity"]).cumcount()
                comps = comps.drop(columns="start").merge(
                    starts[["case", "activity", "occ", "end"]].rename(columns={"end": "start"}),
                    on=["case", "activity", "occ"], how="left").drop(columns="occ")
            df = comps

    df = df.dropna(subset=["end"]).copy()
    df.loc[df["start"] > df["end"], "start"] = pd.NaT
    df["has_start"] = df["start"].notna()
    df["begin"] = df["start"].fillna(df["end"])
    df = df.sort_values(["case", "begin", "end"], kind="mergesort").reset_index(drop=True)

    g = df.groupby("case", sort=False)
    df["pos"] = g.cumcount() + 1
    df["duration_h"] = (df["end"] - df["start"]).dt.total_seconds() / 3600
    df["_cummax_end"] = g["end"].cummax()
    prev_end = df.groupby("case", sort=False)["_cummax_end"].shift()
    df["wait_before_h"] = ((df["begin"] - prev_end).dt.total_seconds() / 3600).clip(lower=0)
    df["prev_activity"] = g["activity"].shift()
    df["next_activity"] = g["activity"].shift(-1)
    df["wait_after_h"] = df.groupby("case", sort=False)["wait_before_h"].shift(-1)
    df["occ"] = df.groupby(["case", "activity"], sort=False).cumcount() + 1
    df["is_repeat"] = df["occ"] > 1
    df["self_loop"] = df["activity"].eq(df["prev_activity"])
    return df.drop(columns="_cummax_end")


def to_pm4py_log(ev):
    """Standard PM4Py dataframe (case:concept:name, concept:name, time:timestamp)."""
    d = ev[["case", "activity", "begin", "end"]].copy()
    return pm4py.format_dataframe(d, case_id="case", activity_key="activity", timestamp_key="begin")


def build_cases(ev):
    """Case table + path (variant) catalogue. Path = ordered sequence of activities of the case."""
    g = ev.groupby("case", sort=False)
    cases = g.agg(first_activity=("activity", "first"), last_activity=("activity", "last"),
                  n_events=("activity", "size"), case_start=("begin", "min"), case_end=("end", "max"))
    cases["variant"] = g["activity"].agg(lambda s: tuple(s))
    cases["case_duration_h"] = (cases.case_end - cases.case_start).dt.total_seconds() / 3600
    # Paths are keyed by a joined string, NOT by tuples: in pandas 2.x, mapping/value_counts on tuple keys
    # turns them into a MultiIndex and fails with "Reindexing only valid with uniquely valued Index objects".
    SEP = "\u241f"
    cases["variant_key"] = cases["variant"].map(lambda t: SEP.join(t))
    vc = cases["variant_key"].value_counts(sort=True)
    vid = pd.Series([f"P{i+1}" for i in range(len(vc))], index=vc.index.astype(str))
    cases["path_id"] = cases["variant_key"].map(vid)
    seq_of = cases.drop_duplicates("variant_key").set_index("variant_key")["variant"]
    variants = pd.DataFrame({"path_id": vid.values, "variant": [seq_of[k] for k in vc.index],
                             "cases": vc.values})
    variants["pct_cases"] = 100 * variants.cases / variants.cases.sum()
    variants["steps"] = variants.variant.map(len)
    variants = variants.set_index("path_id")
    ev["path_id"] = ev["case"].map(cases["path_id"])
    return cases, variants


def apply_partial_rules(cases, exp_start, exp_end, open_days, log_end):
    """Flag partial cases: wrong start, missing expected end; split missing-end into open vs stopped."""
    c = cases
    ms = ~c.first_activity.isin(exp_start) if exp_start else pd.Series(False, index=c.index)
    me = ~c.last_activity.isin(exp_end) if exp_end else pd.Series(False, index=c.index)
    idle_days = (log_end - c.case_end).dt.total_seconds() / 86400
    c["idle_days_at_extract"] = idle_days
    c["is_partial"] = ms | me
    c["partial_reason"] = np.select(
        [ms & me, ms, me & (idle_days <= open_days), me],
        ["missing start & end", "missing start", "open / ongoing", "stopped before end"], "complete")
    return c


def path_label(pid, row, maxlen=110):
    seq = " → ".join(row.variant)
    if len(seq) > maxlen:
        seq = seq[:maxlen] + "…"
    return f"{pid} | {row.cases} cases ({row.pct_cases:.1f}%) | {row.steps} steps | {seq}"

## 0d · Activity-level metric functions
Each function takes the events (`e`) and cases (`c`) of the selected scope (one path or the whole log) and returns one row per activity.

In [5]:
def get_scope(path_id, include_partial=True):
    ev, cases = STATE["ev"], STATE["cases"]
    c = cases if path_id == ALL else cases[cases.path_id == path_id]
    if not include_partial:
        c = c[~c.is_partial]
    return ev[ev.case.isin(c.index)].copy(), c


def parse_rate_overrides(text):
    rates = {}
    for line in (text or "").splitlines():
        mt = re.match(r"^\s*(.+?)\s*[:=]\s*([-+]?\d*\.?\d+)\s*$", line)
        if mt:
            rates[mt.group(1)] = float(mt.group(2))
    return rates


def compute_cost(e, mode, rate, fixed, overrides):
    if mode == "column":
        return e["cost_log"].fillna(0.0)
    r = e["activity"].map(overrides).fillna(rate)
    return fixed + r * e["duration_h"].fillna(0.0)


# Q1 ------------------------------------------------------------------
def activity_frequency(e, c):
    n = max(len(c), 1)
    f = e.groupby("activity").agg(occurrences=("case", "size"), cases_with_activity=("case", "nunique"))
    f["pct_of_cases"] = 100 * f.cases_with_activity / n
    f["avg_per_case"] = f.occurrences / n
    f["share_of_events_pct"] = 100 * f.occurrences / f.occurrences.sum()
    if e["resource"].notna().any():
        f["distinct_resources"] = e.groupby("activity")["resource"].nunique()
    return f.sort_values("occurrences", ascending=False)


# Q2 ------------------------------------------------------------------
def activity_duration(e, fct):
    d = e.dropna(subset=["duration_h"])
    if d.empty:
        return None
    g = d.groupby("activity")["duration_h"]
    t = pd.DataFrame({"executions": g.size(), "mean": g.mean(), "median": g.median(),
                      "p90": g.quantile(0.9), "max": g.max(), "total": g.sum()})
    t["variability_cv"] = g.std() / g.mean()
    for col in ["mean", "median", "p90", "max", "total"]:
        t[col] *= fct
    return t


# Q3 ------------------------------------------------------------------
def activity_waiting(e, fct):
    g = e.groupby("activity")
    wb, wa = g["wait_before_h"], g["wait_after_h"]
    t = pd.DataFrame({
        "waits_measured": wb.count(),
        "mean_wait_before": wb.mean() * fct, "median_wait_before": wb.median() * fct,
        "p90_wait_before": wb.quantile(0.9) * fct, "total_wait_before": wb.sum() * fct,
        "mean_wait_after": wa.mean() * fct, "total_wait_after": wa.sum() * fct})
    h = (e.dropna(subset=["next_activity"])
           .groupby(["activity", "next_activity"])["wait_after_h"]
           .agg(transitions="size", mean_wait="mean", total_wait="sum"))
    h[["mean_wait", "total_wait"]] *= fct
    return t, h.sort_values("total_wait", ascending=False)


# Q4 ------------------------------------------------------------------
def activity_repetition(e):
    per = e.groupby(["activity", "case"]).size()
    g = per.groupby(level="activity")
    r = pd.DataFrame({"cases_with_activity": g.size(), "executions": g.sum(),
                      "cases_with_repeat": g.apply(lambda s: int((s > 1).sum())), "max_per_case": g.max()})
    r["rework_rate_pct"] = 100 * r.cases_with_repeat / r.cases_with_activity
    r["extra_executions"] = r.executions - r.cases_with_activity
    r["repeat_share_of_executions_pct"] = 100 * r.extra_executions / r.executions
    r["self_loops"] = e.groupby("activity")["self_loop"].sum().astype(int)
    if e["duration_h"].notna().any():
        r["rework_time_h"] = e[e.is_repeat].groupby("activity")["duration_h"].sum().reindex(r.index).fillna(0)
    return r.sort_values(["rework_rate_pct", "extra_executions"], ascending=False)


# Q5 ------------------------------------------------------------------
def activity_cost(e, c):
    n = max(len(c), 1)
    g = e.groupby("activity")["cost"]
    t = pd.DataFrame({"executions": g.size(), "total_cost": g.sum(), "mean_cost_per_execution": g.mean(),
                      "cost_per_case": g.sum() / n})
    t["rework_cost"] = e["cost"].where(e["is_repeat"], 0).groupby(e["activity"]).sum()
    tot = t.total_cost.sum()
    t["share_of_total_pct"] = 100 * t.total_cost / tot if tot else 0.0
    return t.sort_values("total_cost", ascending=False)


# Q6 ------------------------------------------------------------------
def partial_association():
    """Log-level: how strongly each activity is associated with partial cases."""
    ev, cases = STATE["ev"], STATE["cases"]
    base = cases.is_partial.mean()
    cw = ev.groupby("activity")["case"].nunique()
    pids = cases.index[cases.is_partial]
    pw = ev[ev.case.isin(pids)].groupby("activity")["case"].nunique().reindex(cw.index, fill_value=0)
    drop = cases.loc[cases.is_partial, "last_activity"].value_counts().reindex(cw.index, fill_value=0)
    t = pd.DataFrame({"cases_with_activity": cw, "partial_cases_with_activity": pw})
    t["pct_partial"] = 100 * pw / cw
    t["lift_vs_log"] = (pw / cw) / base if base > 0 else np.nan
    t["dropoff_cases"] = drop
    t["dropoff_rate_pct"] = 100 * drop / cw
    return t.sort_values("lift_vs_log", ascending=False), base


def abandoned_on_path(path_id):
    """Partial cases whose sequence is a strict prefix of the selected (complete) path."""
    cases, variants = STATE["cases"], STATE["variants"]
    S = variants.loc[path_id, "variant"]
    pc = cases[cases.is_partial]
    mask = pc.variant.map(lambda t: len(t) < len(S) and S[:len(t)] == t)
    return pc[mask], S


# Scorecard -----------------------------------------------------------
FLAG_NAMES = {"occurrences": "frequent", "mean_duration": "slow", "total_wait_before": "queue in front",
              "total_wait_caused": "causes waiting", "rework_rate_pct": "repeats",
              "total_cost": "costly", "partial_lift": "linked to partial"}

def activity_scorecard(e, c, fct):
    fr = activity_frequency(e, c); du = activity_duration(e, fct)
    wa, _ = activity_waiting(e, fct); rp = activity_repetition(e); co = activity_cost(e, c)
    pa, _ = partial_association()
    sc = pd.DataFrame(index=fr.index)
    sc["occurrences"] = fr.occurrences
    sc["mean_duration"] = du["mean"] if du is not None else np.nan
    sc["total_wait_before"] = wa.total_wait_before
    sc["total_wait_caused"] = wa.total_wait_after
    sc["rework_rate_pct"] = rp.rework_rate_pct
    sc["total_cost"] = co.total_cost
    sc["partial_lift"] = pa.lift_vs_log.reindex(sc.index)
    sc = sc.dropna(axis=1, how="all")
    ranks = sc.rank(pct=True)
    sc["hotspot_score"] = (ranks.mean(axis=1) * 100).round(0)
    flags = {a: [] for a in sc.index}
    for col, name in FLAG_NAMES.items():
        if col not in sc:
            continue
        thr = 1.0 if col == "partial_lift" else 0.0
        for a in sc[col][sc[col] > thr].nlargest(3).index:
            flags[a].append(name)
    sc["flags"] = [", ".join(flags[a]) for a in sc.index]
    return sc.sort_values("hotspot_score", ascending=False)


# Path comparison -----------------------------------------------------
PATH_METRICS = ["Executions per case", "Mean processing time", "Mean waiting before",
                "Mean waiting caused (after)", "Rework rate %", "Mean cost per execution"]

def path_activity_matrix(metric, paths, include_partial, cost_args, fct):
    ev, cases = STATE["ev"], STATE["cases"]
    c = cases[cases.path_id.isin(paths)]
    if not include_partial:
        c = c[~c.is_partial]
    e = ev[ev.case.isin(c.index)].copy()
    e["cost"] = compute_cost(e, *cost_args)
    g = e.groupby(["activity", "path_id"])
    if metric == "Executions per case":
        m = g.size().div(c.path_id.value_counts(), level="path_id")
    elif metric == "Mean processing time":
        m = g["duration_h"].mean() * fct
    elif metric == "Mean waiting before":
        m = g["wait_before_h"].mean() * fct
    elif metric == "Mean waiting caused (after)":
        m = g["wait_after_h"].mean() * fct
    elif metric == "Rework rate %":
        per = e.groupby(["activity", "path_id", "case"]).size()
        m = (per > 1).groupby(level=["activity", "path_id"]).mean() * 100
    else:
        m = g["cost"].mean()
    mat = m.unstack("path_id").reindex(columns=[p for p in paths if p in c.path_id.values])
    order = e.activity.value_counts().index
    return mat.reindex([a for a in order if a in mat.index])

## 0e · Display helpers

In [6]:
def hbar(df, x, title, y="activity", color=None, fmt=".1f", vline=None):
    d = df.reset_index()
    if d.empty:
        print("Nothing to plot."); return
    fig = px.bar(d, x=x, y=y, orientation="h", title=title, text=x, color=color,
                 color_continuous_scale="Reds" if color else None)
    fig.update_traces(texttemplate="%{text:" + fmt + "}", textposition="outside", cliponaxis=False)
    fig.update_layout(yaxis=dict(autorange="reversed", title=""), height=max(320, 30 * len(d) + 140),
                      margin=dict(l=10, r=60, t=60, b=10), coloraxis_showscale=False)
    if vline is not None:
        fig.add_vline(x=vline, line_dash="dash", line_color="grey")
    fig.show()


def show_table(df, gradient=None, fmt="{:,.2f}"):
    try:
        import jinja2  # noqa: F401  (needed by DataFrame.style)
    except ImportError:
        display(df.round(2)); return
    num = df.select_dtypes("number").columns
    sty = df.style.format(fmt, subset=num, na_rep="–")
    if gradient:
        cols = [g for g in gradient if g in df.columns and df[g].notna().any()]
        if cols:
            sty = sty.background_gradient(cmap="Reds", subset=cols)
    display(sty)


def note(html, kind="info"):
    colors = {"info": "#eef4fb", "warn": "#fff4e5", "good": "#edf7ed"}
    display(HTML(f"<div style='background:{colors[kind]};padding:8px 12px;border-radius:6px;"
                 f"margin:6px 0;font-size:13px'>{html}</div>"))


def chevrons(seq):
    parts = [f"<span style='display:inline-block;background:#2f5d8a;color:white;padding:4px 10px;"
             f"margin:2px;border-radius:4px;font-size:12px'>{i}. {a}</span>" for i, a in enumerate(seq, 1)]
    return "<div style='line-height:2.2'>" + " <b>→</b> ".join(parts) + "</div>"


def show_dfg(e):
    try:
        log = to_pm4py_log(e)
        dfg, sa, ea = pm4py.discover_dfg(log)
        tmp = os.path.join(tempfile.gettempdir(), "activity_dfg.png")
        pm4py.save_vis_dfg(dfg, sa, ea, tmp)
        display(Image(filename=tmp))
    except Exception as ex:
        print(f"(Process map needs Graphviz – showing directly-follows table instead: {ex})")
        df_rel = e.dropna(subset=["next_activity"]).groupby(["activity", "next_activity"]).size()
        display(df_rel.sort_values(ascending=False).head(25).to_frame("count"))

---
## Step 1 · Load the event log
Enter a file path (recommended for large logs) **or** upload a file **or** tick the demo log, then click **Load log**.

In [7]:
w_file = W.Text(placeholder="e.g. /data/my_log.xes  or  my_log.csv", description="File path:",
                layout=W.Layout(width="65%"))
w_upload = W.FileUpload(accept=".xes,.gz,.csv,.txt,.tsv,.xlsx,.xls,.parquet", multiple=False,
                        description="Upload")
w_sep = W.Dropdown(options=[("comma ( , )", ","), ("semicolon ( ; )", ";"), ("tab", "\t"), ("pipe ( | )", "|")],
                   description="CSV sep:")
w_demo = W.Checkbox(value=False, description="Use built-in demo log (loan applications)", indent=False)
b_load = W.Button(description="1 ▸ Load log", button_style="primary", icon="upload")
out_load = W.Output()

def on_load(_=None):
    with out_load:
        clear_output()
        try:
            if w_demo.value:
                raw, src = make_demo_log(), "demo log"
            elif w_upload.value:
                name, content = _uploaded_file(w_upload)
                raw, src = load_log(name, content=content, sep=w_sep.value), name
            elif w_file.value.strip():
                raw, src = load_log(w_file.value.strip(), sep=w_sep.value), w_file.value.strip()
            else:
                print("Provide a file path, upload a file, or tick the demo log."); return
        except Exception as ex:
            print(f"❌ Could not load the log: {ex}"); return
        dup = raw.columns[raw.columns.duplicated()].tolist()
        if dup:
            print(f"⚠️ Duplicate column names dropped (kept first): {dup}")
        raw = raw.loc[:, ~raw.columns.duplicated()].reset_index(drop=True)
        for k in list(STATE):
            STATE.pop(k)
        STATE["raw"], STATE["source"] = raw, src
        print(f"✅ Loaded {src}: {len(raw):,} rows × {raw.shape[1]} columns")
        display(raw.head(8))
    if "render_mapping_ui" in globals():
        render_mapping_ui()

b_load.on_click(on_load)
display(W.VBox([w_file, W.HBox([w_upload, w_sep]), w_demo, b_load, out_load]))

## Step 2 · Map columns and prepare the event log
Required: case ID, activity, (complete) timestamp. Optional: start timestamp **or** lifecycle column (start/complete pairs),
resource, cost. Paths (variants) are derived from the ordered activity sequence of each case.

In [11]:
mapping_box = W.VBox()
out_prep = W.Output()

def render_mapping_ui():
    raw = STATE.get("raw")
    if raw is None:
        mapping_box.children = [W.HTML("<i>Load a log in Step 1 first.</i>")]; return
    cols = list(raw.columns)
    g = guess_mapping(cols)
    req = [(c, c) for c in cols]
    opt = [("— none —", None)] + req
    st, lay = {"description_width": "150px"}, W.Layout(width="420px")
    dd = {
        "case": W.Dropdown(options=req, value=g["case"], description="Case ID *", style=st, layout=lay),
        "activity": W.Dropdown(options=req, value=g["activity"], description="Activity *", style=st, layout=lay),
        "timestamp": W.Dropdown(options=req, value=g["timestamp"], description="(Complete) timestamp *", style=st, layout=lay),
        "start": W.Dropdown(options=opt, value=g["start"], description="Start timestamp", style=st, layout=lay),
        "lifecycle": W.Dropdown(options=opt, value=g["lifecycle"], description="Lifecycle", style=st, layout=lay),
        "resource": W.Dropdown(options=opt, value=g["resource"], description="Resource", style=st, layout=lay),
        "cost": W.Dropdown(options=opt, value=g["cost"], description="Cost", style=st, layout=lay),
    }
    STATE["map_widgets"] = dd
    b = W.Button(description="2 ▸ Prepare event log", button_style="primary", icon="cogs")
    b.on_click(on_prepare)
    STATE["prep_button"] = b
    mapping_box.children = [W.HBox([W.VBox(list(dd.values())[:4]), W.VBox(list(dd.values())[4:])]), b]

def on_prepare(_=None):
    m = {k: w.value for k, w in STATE["map_widgets"].items()}
    with out_prep:
        clear_output()
        try:
            ev = prepare_events(STATE["raw"], m)
            cases, variants = build_cases(ev)
            log = to_pm4py_log(ev)
        except Exception as ex:
            import traceback
            print(f"❌ Preparation failed: {ex}")
            print("".join(traceback.format_exc().splitlines(keepends=True)[-6:])); return
        STATE.update(mapping=m, ev=ev, cases=cases, variants=variants, pm4py_log=log,
                     log_end=ev["end"].max())
        start_cov = 100 * ev.has_start.mean()
        cost_cov = 100 * ev.cost_log.notna().mean()
        # cross-check variant count with PM4Py
        try:
            n_var_pm4py = len(pm4py.get_variants(log))
        except Exception:
            n_var_pm4py = "n/a"
        print(f"✅ {len(ev):,} activity instances | {len(cases):,} cases | {ev.activity.nunique()} activities | "
              f"{len(variants):,} paths (PM4Py: {n_var_pm4py})")
        print(f"   Period: {ev.begin.min():%Y-%m-%d} → {ev.end.max():%Y-%m-%d}")
        print(f"   Events with start time: {start_cov:.0f}%  | events with cost: {cost_cov:.0f}%")
        if start_cov < 50:
            note("Fewer than half of the events have a start time. Processing time will be limited and "
                 "<b>waiting before</b> will represent <i>elapsed time since the previous event</i>.", "warn")
        top = variants.head(10).copy()
        top["sequence"] = top.variant.map(lambda v: " → ".join(v))
        display(top[["cases", "pct_cases", "steps", "sequence"]].style.format({"pct_cases": "{:.1f}%"}))
    if "render_partial_ui" in globals():
        render_partial_ui()

display(mapping_box, out_prep)
render_mapping_ui()

VBox()

Output()

## Step 3 · Define complete vs partial cases
A case is **partial** if it does not start with an expected start activity or does not end with an expected end activity.
Cases missing the end whose last event is recent (≤ *N* days before the end of the log) are labelled **open / ongoing**;
the rest are **stopped before end** (abandoned, cancelled, or truncated). Defaults come from PM4Py's start/end activity statistics.

In [9]:
partial_box = W.VBox()
out_partial = W.Output()

def render_partial_ui():
    if "cases" not in STATE:
        partial_box.children = [W.HTML("<i>Prepare the event log in Step 2 first.</i>")]; return
    log, n = STATE["pm4py_log"], len(STATE["cases"])
    sa = pm4py.get_start_activities(log)
    ea = pm4py.get_end_activities(log)
    sa_sorted = sorted(sa.items(), key=lambda kv: -kv[1])
    ea_sorted = sorted(ea.items(), key=lambda kv: -kv[1])
    def_s = [a for a, k in sa_sorted if k / n >= 0.10] or [sa_sorted[0][0]]
    def_e = [a for a, k in ea_sorted if k / n >= 0.10] or [ea_sorted[0][0]]
    lay = W.Layout(width="440px")
    w_s = W.SelectMultiple(options=[(f"{a}  ({k} cases, {100*k/n:.0f}%)", a) for a, k in sa_sorted],
                           value=tuple(def_s), rows=min(8, len(sa_sorted)), description="Start acts",
                           layout=lay)
    w_e = W.SelectMultiple(options=[(f"{a}  ({k} cases, {100*k/n:.0f}%)", a) for a, k in ea_sorted],
                           value=tuple(def_e), rows=min(8, len(ea_sorted)), description="End acts",
                           layout=lay)
    w_d = W.IntSlider(value=7, min=0, max=180, description="Open if idle ≤ (days)",
                      style={"description_width": "150px"}, layout=W.Layout(width="440px"))
    b = W.Button(description="3 ▸ Apply & build dashboard", button_style="success", icon="check")
    STATE["partial_widgets"] = (w_s, w_e, w_d)
    b.on_click(on_apply_partial)
    STATE["apply_button"] = b
    partial_box.children = [W.HTML("<small>Ctrl/Cmd-click to select several.</small>"),
                            W.HBox([w_s, w_e]), w_d, b]

def on_apply_partial(_=None):
    w_s, w_e, w_d = STATE["partial_widgets"]
    STATE["cases"] = apply_partial_rules(STATE["cases"], list(w_s.value), list(w_e.value), w_d.value,
                                         STATE["log_end"])
    c = STATE["cases"]
    with out_partial:
        clear_output()
        vc = c.partial_reason.value_counts().to_frame("cases")
        vc["pct"] = 100 * vc.cases / len(c)
        display(vc.style.format({"pct": "{:.1f}%"}))
    if "render_dashboard" in globals():
        render_dashboard()

display(partial_box, out_partial)
render_partial_ui()

VBox()

Output()

## Step 4 · Activity-level dashboard
* **Path** – a single path (variant, ranked by number of cases) or the whole log.
* **Include partial cases** – applies to tabs 1–5 and the scorecard (tab 6 always uses all cases).
* **Cost** – use the log's cost column, or estimate cost = fixed € per execution + hourly rate × processing time
  (per-activity rates can be overridden, one `Activity = rate` per line).
* The **Overview** tab of a single path shows its **step profile**: because every case in a path follows the same sequence,
  waiting and working time can be read step by step.

In [10]:
dash_box = W.VBox()
DASH = {}

def tab_overview(ctx):
    e, c, pid, u = ctx["e"], ctx["c"], ctx["path_id"], ctx["unit"]
    ev_all, cases_all, variants = STATE["ev"], STATE["cases"], STATE["variants"]
    if pid == ALL:
        note(f"<b>Whole log</b>: {len(c):,} cases, {len(e):,} activity instances, {e.activity.nunique()} activities, "
             f"{c.path_id.nunique():,} paths. Partial cases: {100*cases_all.is_partial.mean():.1f}%.")
        v = variants.head(15).copy()
        v["partial_share_pct"] = cases_all.groupby("path_id").is_partial.mean().reindex(v.index) * 100
        v[f"mean_case_duration_{u}"] = cases_all.groupby("path_id").case_duration_h.mean().reindex(v.index) * ctx["f"]
        v["sequence"] = v.variant.map(lambda s: " → ".join(s))
        show_table(v.drop(columns="variant"), gradient=["cases"])
        print("Directly-follows graph (PM4Py) of the selected scope:")
        show_dfg(e)
        return
    row = variants.loc[pid]
    pr = cases_all.loc[cases_all.path_id == pid, "partial_reason"].value_counts()
    status = ("<span style='color:#2e7d32'>complete path</span>" if pr.index.tolist() == ["complete"]
              else "<span style='color:#c62828'>partial path</span> (" + ", ".join(f"{k}: {v}" for k, v in pr.items()) + ")")
    note(f"<b>{pid}</b> – {row.cases} cases ({row.pct_cases:.1f}% of log), {row.steps} steps, "
         f"mean case duration {c.case_duration_h.mean()*ctx['f']:.1f} {u}; {status}")
    display(HTML(chevrons(row.variant)))
    if e.empty:
        print("No cases in scope (are partial cases excluded?)."); return
    sp = (e.groupby(["pos", "activity"])
            .agg(cases=("case", "nunique"), mean_wait_before=("wait_before_h", "mean"),
                 mean_processing=("duration_h", "mean"), mean_cost=("cost", "mean"),
                 resources=("resource", "nunique"))
            .reset_index())
    sp[["mean_wait_before", "mean_processing"]] *= ctx["f"]
    sp["step"] = sp.pos.astype(str) + ". " + sp.activity
    fig = go.Figure()
    fig.add_bar(y=sp.step, x=sp.mean_wait_before.fillna(0), name=f"waiting before ({u})", orientation="h",
                marker_color="#ef9a9a")
    fig.add_bar(y=sp.step, x=sp.mean_processing.fillna(0), name=f"processing ({u})", orientation="h",
                marker_color="#2f5d8a")
    fig.update_layout(barmode="stack", title=f"Step profile of {pid}: where time goes, step by step",
                      yaxis=dict(autorange="reversed"), height=max(320, 34 * len(sp) + 140),
                      margin=dict(l=10, r=20, t=60, b=10), legend=dict(orientation="h", y=-0.15))
    fig.show()
    show_table(sp.drop(columns=["step"]).set_index(["pos", "activity"]),
               gradient=["mean_wait_before", "mean_processing", "mean_cost"])


def tab_frequency(ctx):
    e, c = ctx["e"], ctx["c"]
    f = activity_frequency(e, c)
    top = f.head(ctx["topn"])
    a = f.index[0]
    note(f"<b>{a}</b> is the most frequent activity: {f.loc[a,'occurrences']:,} executions, "
         f"{f.loc[a,'avg_per_case']:.2f} per case. Activities with <b>executions per case &gt; 1</b> are executed "
         f"repeatedly within this path – compare with the Repetition tab.")
    hbar(top, "occurrences", "Executions per activity", color="avg_per_case", fmt=",.0f")
    show_table(f, gradient=["occurrences", "avg_per_case"])


def tab_duration(ctx):
    e, u, stat = ctx["e"], ctx["unit"], ctx["stat"]
    d = activity_duration(e, ctx["f"])
    if d is None:
        note("No start timestamps available, so processing time cannot be measured. "
             "Map a start-timestamp or lifecycle column in Step 2. The Waiting tab shows elapsed time between events.", "warn")
        return
    d = d.sort_values(stat, ascending=False)
    a = d.index[0]
    note(f"<b>{a}</b> takes longest ({stat} {d.loc[a, stat]:.2f} {u}). Total processing time shows where effort accumulates; "
         f"a high <b>variability_cv</b> (&gt;1) means inconsistent ways of working in that activity.")
    hbar(d.head(ctx["topn"]), stat, f"{stat.capitalize()} processing time per execution ({u})", color="total")
    show_table(d, gradient=["mean", "p90", "total", "variability_cv"])


def tab_waiting(ctx):
    e, u = ctx["e"], ctx["unit"]
    t, h = activity_waiting(e, ctx["f"])
    if not e.has_start.any():
        note("Only one timestamp per event: waiting here = elapsed time since the previous event (waiting + processing).", "warn")
    t = t.sort_values("total_wait_after", ascending=False)
    a = t["total_wait_after"].idxmax() if t["total_wait_after"].notna().any() else None
    b = t["mean_wait_before"].idxmax() if t["mean_wait_before"].notna().any() else None
    msg = []
    if a: msg.append(f"<b>{a}</b> causes the most downstream waiting (total {t.loc[a,'total_wait_after']:,.1f} {u} idle after it)")
    if b: msg.append(f"<b>{b}</b> has the longest queue in front of it (mean {t.loc[b,'mean_wait_before']:.2f} {u})")
    note(". ".join(msg) + ". <i>Waiting before</i> = queue in front of the activity; <i>waiting after</i> = idle time "
         "handed to the next activity (the hand-over the activity causes).")
    long = (t[["mean_wait_before", "mean_wait_after"]].head(ctx["topn"]).reset_index()
              .melt(id_vars="activity", var_name="measure", value_name=u))
    fig = px.bar(long, y="activity", x=u, color="measure", barmode="group", orientation="h",
                 title=f"Mean waiting before vs. caused after each activity ({u})",
                 color_discrete_map={"mean_wait_before": "#ef9a9a", "mean_wait_after": "#c62828"})
    fig.update_layout(yaxis=dict(autorange="reversed", title=""), height=max(320, 46 * min(len(t), ctx["topn"]) + 140),
                      margin=dict(l=10, r=20, t=60, b=10))
    fig.show()
    show_table(t, gradient=["mean_wait_before", "total_wait_before", "mean_wait_after", "total_wait_after"])
    print("Hand-overs that generate the most waiting (activity → next activity):")
    show_table(h.head(15), gradient=["total_wait"])


def tab_repetition(ctx):
    e = ctx["e"]
    r = activity_repetition(e)
    rep = r[r.extra_executions > 0]
    if rep.empty:
        note("No activity repeats within a case in this scope. ✔", "good")
    else:
        a = rep.index[0]
        note(f"<b>{a}</b> repeats in {r.loc[a,'rework_rate_pct']:.0f}% of the cases that perform it "
             f"({int(r.loc[a,'extra_executions'])} extra executions). Self-loops = immediate repetition "
             f"(e.g. retries); non-adjacent repeats indicate rework loops.", "warn")
        hbar(rep.head(ctx["topn"]), "rework_rate_pct", "Rework rate: % of cases executing the activity more than once",
             color="extra_executions", fmt=".0f")
    show_table(r, gradient=["rework_rate_pct", "extra_executions", "self_loops"])


def tab_cost(ctx):
    e, c = ctx["e"], ctx["c"]
    k = activity_cost(e, c)
    if ctx["cost_mode"] == "estimate" and not e.has_start.any():
        note("Estimated cost uses processing time, which is unavailable without start timestamps – only the fixed "
             "cost per execution is counted.", "warn")
    if k.total_cost.sum() == 0:
        note("Total cost is zero. Set an hourly rate / fixed cost, or map a cost column.", "warn")
        show_table(k); return
    a = k.index[0]
    note(f"<b>{a}</b> is the costliest activity: {k.loc[a,'total_cost']:,.0f} total "
         f"({k.loc[a,'share_of_total_pct']:.0f}% of path cost, {k.loc[a,'cost_per_case']:,.1f} per case). "
         f"Rework cost = cost of repeated executions – a direct saving if the repetition is removed.")
    d = k.head(ctx["topn"]).reset_index()
    d["first_time_cost"] = d.total_cost - d.rework_cost
    long = d.melt(id_vars="activity", value_vars=["first_time_cost", "rework_cost"], var_name="part", value_name="cost")
    fig = px.bar(long, y="activity", x="cost", color="part", orientation="h", title="Total cost per activity",
                 color_discrete_map={"first_time_cost": "#2f5d8a", "rework_cost": "#c62828"})
    fig.update_layout(yaxis=dict(autorange="reversed", title=""), height=max(320, 30 * len(d) + 140),
                      margin=dict(l=10, r=20, t=60, b=10))
    fig.show()
    show_table(k, gradient=["total_cost", "mean_cost_per_execution", "rework_cost"])


def tab_partial(ctx):
    pid = ctx["path_id"]
    cases = STATE["cases"]
    t, base = partial_association()
    if base == 0:
        note("No partial cases under the current definition (Step 3).", "good"); return
    note(f"Log-wide partial rate: <b>{100*base:.1f}%</b>. <b>Lift</b> &gt; 1 means cases that perform the activity are "
         f"more likely to be partial than the average case; <b>drop-off</b> counts partial cases whose last event is "
         f"the activity (where they stopped).")
    if pid != ALL:
        seq = list(dict.fromkeys(STATE["variants"].loc[pid, "variant"]))
        t = t.reindex(seq)
        pc = cases[cases.path_id == pid]
        if pc.is_partial.all():
            rs = pc.partial_reason.value_counts()
            note(f"<b>{pid} is itself a partial path</b>: its cases stop after <b>{pc.last_activity.iloc[0]}</b> "
                 f"({', '.join(f'{k}: {v}' for k, v in rs.items())}; median idle at extraction "
                 f"{pc.idle_days_at_extract.median():.1f} days).", "warn")
        else:
            ab, S = abandoned_on_path(pid)
            if ab.empty:
                note(f"No partial cases abandoned along the prefix of {pid}.", "good")
            else:
                rate = 100 * len(ab) / (len(ab) + (~pc.is_partial).sum())
                st = (ab.assign(step=ab.variant.map(len))
                        .groupby(["step", "partial_reason"]).size().unstack(fill_value=0))
                st.index = [f"{i}. {S[i-1]}" for i in st.index]
                st.index.name = "stopped after step"
                note(f"<b>{len(ab)}</b> partial cases followed {pid}'s sequence and stopped part-way "
                     f"(abandonment ≈ <b>{rate:.1f}%</b> of cases entering this path). "
                     f"Note: a prefix shared with other paths is attributed to each of them.", "warn")
                fig = px.bar(st.reset_index().melt(id_vars="stopped after step", var_name="reason", value_name="cases"),
                             y="stopped after step", x="cases", color="reason", orientation="h",
                             title=f"Where cases on {pid} stopped")
                fig.update_layout(yaxis=dict(autorange="reversed", title=""), height=max(300, 40 * len(st) + 140),
                                  margin=dict(l=10, r=20, t=60, b=10))
                fig.show()
                show_table(st, fmt="{:,.0f}")
    hbar(t.head(ctx["topn"]) if pid == ALL else t, "lift_vs_log", "Lift of partial-case probability per activity",
         color="dropoff_cases", fmt=".2f", vline=1)
    show_table(t, gradient=["pct_partial", "lift_vs_log", "dropoff_rate_pct"])


def tab_scorecard(ctx):
    sc = activity_scorecard(ctx["e"], ctx["c"], ctx["f"])
    note(f"Each dimension is ranked (percentile) across activities; <b>hotspot_score</b> is the average rank (0–100). "
         f"<b>flags</b> mark the top-3 activities per question. Units: time in {ctx['unit']}. "
         f"Partial lift is log-wide.")
    show_table(sc, gradient=[c for c in sc.columns if c not in ("flags",)])


def tab_paths(ctx):
    variants = STATE["variants"]
    paths = list(variants.index[: ctx["n_paths"]])
    mat = path_activity_matrix(ctx["path_metric"], paths, ctx["incl"], ctx["cost_args"], ctx["f"])
    note(f"Same activity, different paths: <b>{ctx['path_metric']}</b> for the top {len(paths)} paths "
         f"(blank = activity not on that path). Use this to see whether an activity behaves differently depending "
         f"on the route the case took.")
    if mat.empty:
        print("Nothing to show."); return
    fig = px.imshow(mat, text_auto=".1f", aspect="auto", color_continuous_scale="Reds",
                    title=f"{ctx['path_metric']} by activity × path")
    fig.update_layout(height=max(360, 36 * len(mat) + 160), margin=dict(l=10, r=20, t=60, b=10))
    fig.show()
    show_table(mat.reset_index().set_index("activity"))


TABS = [("Overview", tab_overview), ("1 · Frequency", tab_frequency), ("2 · Duration", tab_duration),
        ("3 · Waiting", tab_waiting), ("4 · Repetition", tab_repetition), ("5 · Cost", tab_cost),
        ("6 · Partial cases", tab_partial), ("Scorecard", tab_scorecard), ("Path comparison", tab_paths)]


def build_ctx():
    d = DASH
    pid = d["path"].value
    cost_args = (d["cost_mode"].value, d["rate"].value, d["fixed"].value, parse_rate_overrides(d["overrides"].value))
    e, c = get_scope(pid, d["incl"].value)
    e["cost"] = compute_cost(e, *cost_args)
    unit = d["unit"].value
    return dict(path_id=pid, e=e, c=c, unit=unit, f=UNITS[unit], stat=d["stat"].value, topn=d["topn"].value,
                incl=d["incl"].value, cost_mode=d["cost_mode"].value, cost_args=cost_args,
                path_metric=d["path_metric"].value, n_paths=d["n_paths"].value)


def refresh(*_):
    i = DASH["tabs"].selected_index or 0
    out = DASH["outs"][i]
    with out:
        clear_output(wait=True)
        try:
            ctx = build_ctx()
            if ctx["c"].empty and TABS[i][1] not in (tab_partial, tab_paths):
                print("No cases in scope – this path may consist only of partial cases; tick 'Include partial cases'.")
                return
            TABS[i][1](ctx)
        except Exception as ex:
            import traceback; traceback.print_exc()


def export_excel(_=None):
    ctx = build_ctx()
    e, c, f = ctx["e"], ctx["c"], ctx["f"]
    name = f"activity_analysis_{'ALL' if ctx['path_id']==ALL else ctx['path_id']}.xlsx"
    with pd.ExcelWriter(name) as xw:
        activity_frequency(e, c).to_excel(xw, sheet_name="1_frequency")
        d = activity_duration(e, f)
        if d is not None: d.to_excel(xw, sheet_name="2_duration")
        w, h = activity_waiting(e, f); w.to_excel(xw, sheet_name="3_waiting"); h.to_excel(xw, sheet_name="3_handovers")
        activity_repetition(e).to_excel(xw, sheet_name="4_repetition")
        activity_cost(e, c).to_excel(xw, sheet_name="5_cost")
        partial_association()[0].to_excel(xw, sheet_name="6_partial")
        activity_scorecard(e, c, f).to_excel(xw, sheet_name="scorecard")
    with DASH["msg"]:
        clear_output(); print(f"💾 Saved {os.path.abspath(name)} (time unit: {ctx['unit']})")


def render_dashboard():
    if "cases" not in STATE or "is_partial" not in STATE["cases"]:
        dash_box.children = [W.HTML("<i>Complete Steps 1–3 first.</i>")]; return
    variants = STATE["variants"]
    opts = [("All paths (whole log)", ALL)] + [(path_label(p, r), p) for p, r in variants.head(500).iterrows()]
    st = {"description_width": "110px"}
    has_cost = STATE["ev"].cost_log.notna().any()
    DASH.update(
        path=W.Dropdown(options=opts, value=opts[1][1] if len(opts) > 1 else ALL, description="Path",
                        style=st, layout=W.Layout(width="98%")),
        unit=W.Dropdown(options=list(UNITS), value="hours", description="Time unit", style=st),
        stat=W.Dropdown(options=["mean", "median", "p90"], value="mean", description="Duration stat", style=st),
        topn=W.IntSlider(value=15, min=3, max=50, description="Top N acts", style=st),
        incl=W.Checkbox(value=True, description="Include partial cases", indent=False),
        cost_mode=W.Dropdown(options=([("Cost column from log", "column")] if has_cost else []) +
                             [("Estimate: fixed + rate × processing time", "estimate")],
                             description="Cost basis", style=st, layout=W.Layout(width="420px")),
        rate=W.FloatText(value=50.0, description="Hourly rate", style=st),
        fixed=W.FloatText(value=0.0, description="Fixed / exec.", style=st),
        overrides=W.Textarea(placeholder="Per-activity hourly rate, e.g.\nManager Approval = 120",
                             description="Rate overrides", style=st, layout=W.Layout(width="420px", height="70px")),
        path_metric=W.Dropdown(options=PATH_METRICS, description="Compare by", style=st, layout=W.Layout(width="360px")),
        n_paths=W.IntSlider(value=min(8, len(variants)), min=1, max=max(1, min(30, len(variants))),
                            description="Top paths", style=st),
        msg=W.Output(),
    )
    DASH["outs"] = [W.Output() for _ in TABS]
    tabs = W.Tab(children=DASH["outs"])
    for i, (t, _) in enumerate(TABS):
        tabs.set_title(i, t)
    DASH["tabs"] = tabs
    for k in ["path", "unit", "stat", "topn", "incl", "cost_mode", "rate", "fixed", "overrides", "path_metric", "n_paths"]:
        DASH[k].observe(refresh, names="value")
    tabs.observe(refresh, names="selected_index")
    b_exp = W.Button(description="Export tables to Excel", icon="download")
    b_exp.on_click(export_excel)
    DASH["export"] = b_exp
    controls = W.VBox([
        DASH["path"],
        W.HBox([W.VBox([DASH["unit"], DASH["stat"], DASH["topn"], DASH["incl"]]),
                W.VBox([DASH["cost_mode"], DASH["rate"], DASH["fixed"]]),
                W.VBox([DASH["overrides"], W.HBox([b_exp]), DASH["msg"]])]),
        W.HBox([DASH["path_metric"], DASH["n_paths"]]),
    ])
    dash_box.children = [W.HTML("<h3 style='margin:4px 0'>Activity-level analysis</h3>"), controls, tabs]
    refresh()

display(dash_box)
render_dashboard()

VBox()

---
### Interpretation guide
* **Frequency × duration** = workload. A short but very frequent activity can consume more capacity than a long rare one – check *total* processing time.
* **Waiting before** high → capacity/queue problem *at* that activity (resources, batching, availability).
  **Waiting after** high → the activity hands work over badly (missing information, batching of outputs, approvals parked).
* **Repetition**: non-adjacent repeats are rework loops (e.g. *Check → Request info → Check*); self-loops are retries or data-entry corrections.
  Rework cost shows the direct prize of first-time-right.
* **Partial cases**: high lift + high drop-off at an activity means cases tend to die there (customer drop-out, cancellations);
  "open / ongoing" are not failures – they are simply not finished at extraction time.
* **Path comparison**: if an activity is slow or repeats only in some paths, the root cause is path-specific (e.g. a different team, product, or entry channel) –
  feed this back into the path-level analysis.